# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- build best-quality embeddings
- train Logistic Regression
- train Linear SVM
- compare model metrics
- zip outputs for download

It does **not** collect data from APIs or repositories.

## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [ ]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

## 3. Clone Or Pull Latest Main Branch

In [ ]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

## 4. Copy Uploaded Raw Data Into Backend

In [ ]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

## 5. Install Dependencies

This uses a Kaggle-friendly install. It avoids installing the full pinned backend requirements because those upgrade pandas/numpy and can conflict with Kaggle's preinstalled packages.

In [ ]:
%cd /kaggle/working/code/backend
!pip install "dagster==1.13.16" "dagster-webserver==1.13.16" "rapidfuzz==3.14.3" "psycopg[binary]>=3.2,<4"
!pip install -e dagster-quickstart --no-deps
!python -m dagster --version

import pandas as pd
import sklearn
import pyarrow as pa
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("pyarrow", pa.__version__)

## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [ ]:
%cd /kaggle/working/code/backend/dagster-quickstart
!python -m dagster job execute \
  -m dagster_quickstart.definitions \
  -j researchlanka_no_collection_preprocessing_job

## 7. Verify Preprocessing Outputs

In [ ]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv
!python - <<'PY'
import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))
PY

## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [ ]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

## 9. Train Best-Quality Logistic Regression

In [ ]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000
!cat data/models/linear_svm_primary_domain_metrics.txt

## 11. Compare Models

In [ ]:
%cd /kaggle/working/code/backend
!grep -E "model_family|label_column|accuracy|macro_f1|weighted_f1" data/models/*metrics.txt || true

## 12. Zip Outputs For Download

In [ ]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip

Download this file from the Kaggle output panel:

```text
/kaggle/working/researchlanka-kaggle-outputs.zip
```